# Agentic AI MCP - Server Only

This notebook starts an MCP server that exposes tools.
Run this on the machine where you want to host the tools.

Another machine can connect to this server using the client notebook.

## Step 1: Setup

In [1]:
# install if needed
# !uv pip install agentic_ai_mcp==0.6.4

import agentic_ai_mcp

print(f"Version: {agentic_ai_mcp.__version__}")

Version: 0.6.4


## Step 2: Define Your Functions (to be used as MCP tools)

In [2]:
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b
    
def greet(name: str, times: int = 1) -> str:
    """Greet someone."""
    return ("Hello, " + name + "! ") * times

# This function returns image data to the client
# The return dict must contain:
#   - has_image (bool): True to signal image content is present
#   - list_image (list): one dict per image with keys: data (base64 str), format, width, height
#   - output (str): optional text description of the result
def do_imaging(width: int, height: int) -> dict:
    """Generate a random image and return it with base64 encoded data.
    
    Requires numpy and Pillow to be installed.
    
    Returns dict with has_image=True, list_image=[...], and output=<text result>
    """
    import base64
    from io import BytesIO

    import numpy as np
    from PIL import Image

    # Generate random RGB image
    img_array = np.random.randint(0, 256, (height, width, 3), dtype=np.uint8)

    # Convert to PNG bytes
    img = Image.fromarray(img_array)
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    img_bytes = buffer.getvalue()

    # Encode as base64
    img_base64 = base64.b64encode(img_bytes).decode("utf-8")

    return {
        "has_image": True,
        "list_image": [
            {
                "data": img_base64,
                "format": "png",
                "width": width,
                "height": height
            }
        ],
        "output": f"Generated a {width}x{height} random image"
    }

## Step 3: Create AgenticAIServer and Register Tools

Configure the host/port.

In [3]:
from agentic_ai_mcp import AgenticAIServer

server = AgenticAIServer(host="0.0.0.0", port=8888)

# register your tools
server.register_tool(add)
server.register_tool(multiply)
server.register_tool(greet)
server.register_tool(do_imaging)

print(f"Tools: {server.tools}")
print(f"URL: {server.mcp_url}")

Tools: ['add', 'multiply', 'greet', 'do_imaging']
URL: http://0.0.0.0:8888/mcp


## Step 4: Start the MCP Server

This will start the server in background. The server URL will be:
- Local: `http://127.0.0.1:8888/mcp`
- Remote: `http://<your-ip>:8888/mcp`

Give this URL to the client notebook.

In [4]:
# get your IP for remote connections
import socket

hostname = socket.gethostname()
try:
    local_ip = socket.gethostbyname(hostname)
except:
    local_ip = "<your-ip>"

# Start server in background
server.start()

print("\nServer is ready!")
print("Local URL:  http://127.0.0.1:8888/mcp")
print(f"Remote URL: http://{local_ip}:8888/mcp")

/home/cloud/.local/share/uv/python/cpython-3.13.9-linux-x86_64-gnu/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=3252033) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
INFO:     Started server process [3252096]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8888 (Press CTRL+C to quit)



Server is ready!
Local URL:  http://127.0.0.1:8888/mcp
Remote URL: http://127.0.1.1:8888/mcp


## Step 5: Stop the Server

Run this cell to stop the server when you're done.

In [7]:
server.stop()